# Week 24, Day 4: The Ornstein-Uhlenbeck process and stochastic differential equations

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w024/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Variance relaxes to σ²τ/2

Simulate 20,000 OU paths at once with tau = 10 ms and sigma = 1 (Euler-Maruyama, dt = 0.01 ms) from x = 0 for 60 ms. Store the variance across paths at t = 10 ms in `var_tau` and at 60 ms in `var_end`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
tau, sigma, dt = 10.0, 1.0, 0.01
r_o = np.random.default_rng(1930)
x = np.zeros(20000); var_t = []
for k in range(1, 6001):
    x = ...
    var_t.append(x.var())
var_t = np.array(var_t)
var_tau, var_end = var_t[999], var_t[-1]
print(f"variance at t = tau: {var_tau:.3f} (theory {5 * (1 - np.exp(-2)):.3f});  at 60 ms: {var_end:.3f} (theory 5)")
ts = np.arange(1, 6001) * dt
plt.plot(ts, var_t); plt.plot(ts, 5 * (1 - np.exp(-2 * ts / tau)), "--"); plt.xlabel("ms"); plt.ylabel("variance"); plt.show()

In [ ]:
import numpy as np
def test_relax():
    "the variance at t = tau is within 5% of (sigma^2 tau / 2)(1 - e^-2)"
    assert abs(var_tau - 5 * (1 - np.exp(-2))) / (5 * (1 - np.exp(-2))) < 0.05

def test_stationary():
    "the variance at t = 60 ms is within 5% of sigma^2 tau / 2 = 5"
    assert abs(var_end - 5) / 5 < 0.05

test_relax()
test_stationary()
print("All checks passed")

### Exercise: The correlation time

Simulate one long OU path (tau = 10 ms, sigma = 1, dt = 0.1 ms, 400,000 steps, starting from a draw from the stationary distribution). Compute its autocorrelation `acf` at lags of 0 to 30 ms (0 to 300 steps), and fit the slope of log acf over lags 0 to 20 ms to get `tau_fit`.

In [ ]:
r_a = np.random.default_rng(1931)
dt2, n2 = 0.1, 400000
noise = r_a.normal(size=n2) * sigma * np.sqrt(dt2)
xs = np.empty(n2); xs[0] = r_a.normal() * np.sqrt(5)
for k in range(1, n2):
    xs[k] = xs[k - 1] - xs[k - 1] / tau * dt2 + noise[k]
xc = xs - xs.mean()
acf = np.array([np.mean(xc[:n2 - L] * xc[L:]) for L in range(301)]) / np.mean(xc ** 2)
lags = np.arange(301) * dt2
tau_fit = ...
print(f"autocorrelation at 10 ms {acf[100]:.3f} (e^-1 = {np.exp(-1):.3f});  fitted correlation time {tau_fit:.2f} ms")
plt.plot(lags, acf); plt.plot(lags, np.exp(-lags / tau), "--"); plt.xlabel("lag (ms)"); plt.ylabel("autocorrelation"); plt.show()

In [ ]:
import numpy as np
def test_tau():
    "the fitted correlation time is within 10% of tau = 10 ms"
    assert abs(tau_fit - 10) / 10 < 0.10

def test_acf1():
    "the autocorrelation at lag 10 ms is within 0.05 of e^-1"
    assert abs(acf[100] - np.exp(-1)) < 0.05

test_tau()
test_acf1()
print("All checks passed")

### Exercise: Sampling a double well

For U(x) = 2(x² − 1)², run 4,000 Langevin chains at once from x = 0 (dt = 0.001, 5,000 steps) and store their final positions in `samples`. Compare a 30-bin histogram on [−2, 2] with the target density (exp(−U) normalised on the same bins) and store the total variation distance, half the sum of absolute differences in bin probabilities, in `tv`.

In [ ]:
U_grad = lambda x: 8 * x * (x ** 2 - 1)
r_l = np.random.default_rng(1908)
x = np.zeros(4000); dtl = 0.001
for _ in range(5000):
    x = ...
samples = x
edges = np.linspace(-2, 2, 31); mids = (edges[:-1] + edges[1:]) / 2
p_emp = np.histogram(samples, edges)[0] / len(samples)
p_true = np.exp(-2 * (mids ** 2 - 1) ** 2); p_true /= p_true.sum()
tv = ...
print(f"total variation distance {tv:.3f};  fraction in the right-hand well {np.mean(samples > 0):.3f}")
plt.bar(mids, p_emp, width=edges[1] - edges[0], alpha=0.6, label="Langevin samples"); plt.plot(mids, p_true, "k", label="target")
plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_tv():
    "the histogram of the chains matches the target density with total variation distance below 0.05"
    assert tv < 0.05

def test_both():
    "both wells are populated, each with between 40% and 60% of the samples"
    assert 0.4 < np.mean(samples > 0) < 0.6

test_tv()
test_both()
print("All checks passed")